STEP 1 

Import Libraries / Завантаження бібліотек

In [3]:
import os
import urllib.parse
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Set display options for DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("✅ All libraries successfully imported!")

✅ All libraries successfully imported!


Database Connection & Data Loading / Підключення та завантаження

In [4]:
# Load environment variables
load_dotenv('../.env')

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME", "lianes_library")

# Encode password safely for connection string
encoded_password = urllib.parse.quote_plus(DB_PASSWORD) if DB_PASSWORD else ""

# Create SQLAlchemy engine
connection_string = f"mysql+mysqlconnector://{DB_USER}:{encoded_password}@{DB_HOST}/{DB_NAME}"
engine = create_engine(connection_string)

# Read tables into Pandas DataFrames
df_books = pd.read_sql("SELECT * FROM books;", engine)
df_friends = pd.read_sql("SELECT * FROM friends;", engine)
df_loans = pd.read_sql("SELECT * FROM loans;", engine)

print("✅ Data successfully loaded from MySQL!")
print(f"Books count: {len(df_books)}, Friends count: {len(df_friends)}, Loans count: {len(df_loans)}")

✅ Data successfully loaded from MySQL!
Books count: 5, Friends count: 3, Loans count: 3


Initial Data Inspection / Швидкий перегляд - Перевірка завантажених даних

In [5]:
# Preview the first few rows of each DataFrame
display(df_books.head(3))
display(df_friends.head(3))
display(df_loans.head(3))

,book_id,title,author,genre,isbn,notes
0,1,The Great Gatsby,F. Scott Fitzgerald,Classic,9780743273565,Good condition
1,2,1984,George Orwell,Dystopian,9780451524935,Favorite
2,3,To Kill a Mockingbird,Harper Lee,Classic,9780061120084,Hardcover


,friend_id,name,email,phone
0,1,Sarah Miller,sarah.m@example.com,+1234567890
1,2,John Doe,john.doe@example.com,+1987654321
2,3,Emma Watson,emma.w@example.com,+1122334455


,loan_id,book_id,friend_id,loan_date,due_date,return_date,notes
0,1,1,1,2026-06-01,2026-06-15,2026-06-14,Returned on time
1,2,2,2,2026-07-01,2026-07-15,None,"Currently borrowed, overdue"
2,3,4,3,2026-07-10,2026-07-24,None,Currently borrowed


STEP 2 - EXPLORATORY DATA ANALYSIS (EDA)

In [6]:
# Check data types and non-null counts for each DataFrame
print("--- Books Info ---")
df_books.info()

print("\n--- Friends Info ---")
df_friends.info()

print("\n--- Loans Info ---")
df_loans.info()

# What this code does: It shows how many rows are in each table, 
# what columns there are, and which data types (numbers, strings, dates) Python uses.

--- Books Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   book_id  5 non-null      int64 
 1   title    5 non-null      object
 2   author   5 non-null      object
 3   genre    5 non-null      object
 4   isbn     5 non-null      object
 5   notes    5 non-null      object
dtypes: int64(1), object(5)
memory usage: 368.0+ bytes

--- Friends Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   friend_id  3 non-null      int64 
 1   name       3 non-null      object
 2   email      3 non-null      object
 3   phone      3 non-null      object
dtypes: int64(1), object(3)
memory usage: 224.0+ bytes

--- Loans Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 7 columns):

In [7]:
# Check for missing values in all DataFrames
print("Missing values in Books:\n", df_books.isnull().sum())
print("\nMissing values in Friends:\n", df_friends.isnull().sum())
print("\nMissing values in Loans:\n", df_loans.isnull().sum())

# Check for duplicate rows
print("\nDuplicate rows in Books:", df_books.duplicated().sum())
print("Duplicate rows in Friends:", df_friends.duplicated().sum())
print("Duplicate rows in Loans:", df_loans.duplicated().sum())


# What this code does: Checks whether there are missing values in critical fields 
# and if there are accidentally duplicated rows.

Missing values in Books:
 book_id    0
title      0
author     0
genre      0
isbn       0
notes      0
dtype: int64

Missing values in Friends:
 friend_id    0
name         0
email        0
phone        0
dtype: int64

Missing values in Loans:
 loan_id        0
book_id        0
friend_id      0
loan_date      0
due_date       0
return_date    2
notes          0
dtype: int64

Duplicate rows in Books: 0
Duplicate rows in Friends: 0
Duplicate rows in Loans: 0


In [8]:
# Data Type Conversion for Dates (Data Cleaning)
# In the loans table, dates were loaded as regular text strings (object). 
# For analysis, we need to convert them into datetime format.

# Convert date columns to datetime objects
date_cols = ['loan_date', 'due_date', 'return_date']

for col in date_cols:
    df_loans[col] = pd.to_datetime(df_loans[col])

# Verify converted data types
print("Updated column types for Loans:")
print(df_loans.dtypes)

Updated column types for Loans:
loan_id                 int64
book_id                 int64
friend_id               int64
loan_date      datetime64[ns]
due_date       datetime64[ns]
return_date    datetime64[ns]
notes                  object
dtype: object


STEP 3 - SQL Analytics and CRUD Functions

In [9]:
# READ: Writing SQL queries for analytics

# Helper function to execute SQL queries and return DataFrames
def run_query(query):
    return pd.read_sql(query, engine)

# Query 1: Find all currently borrowed books (not yet returned)
query_borrowed = """
SELECT 
    b.title,
    b.author,
    f.name AS borrowed_by,
    l.loan_date,
    l.due_date
FROM loans l
JOIN books b ON l.book_id = b.book_id
JOIN friends f ON l.friend_id = f.friend_id
WHERE l.return_date IS NULL;
"""

print("--- Currently Borrowed Books ---")
display(run_query(query_borrowed))


# What this code does: Finds all books currently borrowed by 
#  (where return_date IS NULL) and displays the friend's name along 
#  with the due date for returning the book.

--- Currently Borrowed Books ---


,title,author,borrowed_by,loan_date,due_date
0,1984,George Orwell,John Doe,2026-07-01,2026-07-15
1,Sapiens: A Brief History of Humankind,Yuval Noah Harari,Emma Watson,2026-07-10,2026-07-24


In [10]:
# CREATE: Adding a new book function (Insert Data)

from sqlalchemy import text

# Function to add a new book into the database
def add_new_book(title, author, genre, isbn, notes=""):
    insert_query = text("""
        INSERT INTO books (title, author, genre, isbn, notes)
        VALUES (:title, :author, :genre, :isbn, :notes);
    """)
    with engine.connect() as conn:
        conn.execute(insert_query, {
            "title": title, 
            "author": author, 
            "genre": genre, 
            "isbn": isbn, 
            "notes": notes
        })
        conn.commit()
    print(f"✅ Book '{title}' successfully added to the database!")

# Test the function by adding a new book
add_new_book("Dune", "Frank Herbert", "Sci-Fi", "9780441172719", "Classics SF")

# Verify that the book was added
display(run_query("SELECT * FROM books WHERE title = 'Dune';"))


# What this code does: Implements the Create operation from CRUD. 
# It adds a new book to MySQL via Python and immediately verifies that it appeared in the database.

✅ Book 'Dune' successfully added to the database!


,book_id,title,author,genre,isbn,notes
0,6,Dune,Frank Herbert,Sci-Fi,9780441172719,Classics SF


In [11]:
# CREATE: Adding a new friend function (Insert Data)

# Function to add a new friend into the database
def add_new_friend(name, email, phone):
    insert_query = text("""
        INSERT INTO friends (name, email, phone)
        VALUES (:name, :email, :phone);
    """)
    with engine.connect() as conn:
        conn.execute(insert_query, {
            "name": name, 
            "email": email, 
            "phone": phone
        })
        conn.commit()
    print(f"✅ Friend '{name}' successfully added to the database!")

# Test the function by adding a new friend
add_new_friend("Alex Turner", "alex.t@example.com", "+1122334455")

# Verify that the friend was added
display(run_query("SELECT * FROM friends WHERE name = 'Alex Turner';"))

✅ Friend 'Alex Turner' successfully added to the database!


,friend_id,name,email,phone
0,4,Alex Turner,alex.t@example.com,+1122334455


In [12]:
# UPDATE: Book return function (Mark as Returned)

# Function to return a borrowed book
def return_book(loan_id, return_date):
    update_query = text("""
        UPDATE loans
        SET return_date = :return_date, notes = 'Returned'
        WHERE loan_id = :loan_id;
    """)
    with engine.connect() as conn:
        conn.execute(update_query, {"return_date": return_date, "loan_id": loan_id})
        conn.commit()
    print(f"✅ Loan ID {loan_id} updated with return date {return_date}!")

# Test updating loan_id 2 (making it returned today)
return_book(loan_id=2, return_date="2026-07-23")

# Verify update
display(run_query("SELECT * FROM loans WHERE loan_id = 2;"))

# What this code does: Implements the Update operation from CRUD. 
# It marks a book as returned in the database.

✅ Loan ID 2 updated with return date 2026-07-23!


,loan_id,book_id,friend_id,loan_date,due_date,return_date,notes
0,2,2,2,2026-07-01,2026-07-15,2026-07-23,Returned
